# NaverCafe Trend Analysis
Extracts trending topics from community posts (excluding brand/product discussions)
using GPT-4. Results are stored weekly in the data warehouse.

**Pipeline steps:**
1. Load existing trend records to find latest processed date
2. Fetch new posts since last run
3. Filter out brand-related posts
4. Call GPT-4 to extract 3 trend keywords per post
5. Upload results to data warehouse

In [ ]:
# pip install torch azure-storage-blob snowflake-connector-python snowflake-sqlalchemy==1.5.1 openai==0.28

In [ ]:
import os
import re
import time
import pandas as pd
import datetime
import openai
import snowflake.connector
from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta
from snowflake.sqlalchemy import URL
from sqlalchemy import create_engine
from snowflake.connector.pandas_tools import pd_writer

# Current time in KST (UTC+9)
now = datetime.now() + timedelta(hours=9)

In [ ]:
# ── DATABASE CONNECTION ──────────────────────────────────────────────────
# Set environment variables before running:
#   SNOWFLAKE_USER, SNOWFLAKE_PASSWORD, SNOWFLAKE_ACCOUNT
#   SNOWFLAKE_DATABASE, SNOWFLAKE_SCHEMA, SNOWFLAKE_WAREHOUSE
#   OPENAI_API_KEY

con = snowflake.connector.connect(
    user=os.environ['SNOWFLAKE_USER'],
    password=os.environ['SNOWFLAKE_PASSWORD'],
    account=os.environ['SNOWFLAKE_ACCOUNT'],
    database=os.environ['SNOWFLAKE_DATABASE'],
    schema=os.environ['SNOWFLAKE_SCHEMA'],
    warehouse=os.environ['SNOWFLAKE_WAREHOUSE']
)
cur = con.cursor()

In [ ]:
# ── LOAD EXISTING RECORDS TO FIND LATEST DATE ────────────────────────────
orig_query = "SELECT * FROM TREND_TABLE;"
orig_columns = ['SEQ', 'DATE', 'START_WEEK', 'KEYWORD', 'TITLE', 'CONTENTS',
                'POST_NUMBER', 'URL', 'VIEWS']
orig_content = pd.DataFrame(cur.execute(orig_query).fetchall(), columns=orig_columns)
latest_date = orig_content['DATE'].max()
print(f'Latest processed date: {latest_date}')

In [ ]:
# ── FETCH NEW POSTS SINCE LAST RUN ───────────────────────────────────────
# Excludes deal boards and promotional content

content_query = f"""
SELECT *
FROM COMMUNITY_POSTS_VIEW
WHERE WDATE > '{latest_date}'
AND NOT (
   BOARD_PATH IN ('Deal Board')
   OR BOARD_PATH LIKE '%HotDeal%'
   OR TITLE LIKE '%postpartum care%'
   OR TITLE LIKE '%baby products%'
   OR TITLE LIKE '%review%'
   OR TITLE LIKE '%hot deal%'
);
"""
content = cur.execute(content_query)

if not content:
    print('No new data to process.')
else:
    content_columns = ['SEQ', 'POST_NUMBER', 'BOARD_PATH', 'CAFE_NAME', 'TITLE',
                       'CONTENTS', 'MEDIA', 'AUTHOR', 'URL', 'WDATE', 'RDATE',
                       'VIEWS', 'COMMENT_CNT', 'LIKES']
    content = pd.DataFrame(content, columns=content_columns)
    content['CONTENTS'] = content['CONTENTS'].fillna('')
    content['RDATE'] = pd.to_datetime(content['RDATE'], format='%Y-%m-%d').dt.strftime('%Y-%m-%d')
    content['WDATE'] = pd.to_datetime(content['WDATE'], format='%Y-%m-%d').dt.strftime('%Y-%m-%d')
    content['YEAR'] = pd.to_datetime(content['WDATE']).dt.year
    content['MONTH'] = pd.to_datetime(content['WDATE']).dt.month

In [ ]:
# ── FILTER OUT BRAND-RELATED POSTS ───────────────────────────────────────
# Posts mentioning specific brands are excluded from trend analysis
# to capture organic consumer topics beyond product discussions

BRAND_KEYWORDS = {
    'Huggies':  '|'.join(['하기스','huggies','맥스드라이','매직드라이','네이처메이드','naturemade','매직컴포트']),
    'Pampers':  '|'.join(['펨퍼스','팸퍼스','아르모니','베이비드라이','터치오브네이처','에어차차']),
    'Penelope': '|'.join(['페넬로페','미라클올데이','씬씬씬']),
    'Mamipoko': '|'.join(['마미포코','땀먹는팬티','에어핏공기솔솔','리프가닉']),
    'Bosomi':   '|'.join(['보솜이','원더바이원더','메가드라이']),
    'Kindoh':   '|'.join(['킨도','업앤플레이'])
}

def detect_brands(content_text, title):
    return [
        brand for brand, pattern in BRAND_KEYWORDS.items()
        if re.search(pattern, content_text, re.IGNORECASE) or re.search(pattern, title, re.IGNORECASE)
    ]

content['BRANDS'] = content.apply(lambda row: detect_brands(row['CONTENTS'], row['TITLE']), axis=1)

# Keep only posts with no brand mentions, then select top 100 by views per day
content_trend = content[content['BRANDS'].apply(lambda x: len(x) == 0)].drop(columns=['BRANDS'])
content_trend = content_trend.groupby('WDATE').apply(lambda x: x.nlargest(100, 'VIEWS')).reset_index(drop=True)
print(f'Posts selected for trend analysis: {len(content_trend)}')

In [ ]:
# ── GPT-4 TREND KEYWORD EXTRACTION ──────────────────────────────────────
# For each post, GPT-4 extracts 3 trending topic keywords
# unrelated to diapers, pregnancy, or parenting

openai.api_key = os.environ['OPENAI_API_KEY']
GPT_MODEL = 'gpt-4-turbo-2024-04-09'

def extract_trend(title_, content_):
    """Call GPT-4 to extract 3 trending keywords from a community post."""
    prompt = (
        f'Title: {title_}\nContent: {content_}\n'
        'Prompt: The above is a community post. '
        'Excluding topics about diapers, pregnancy, childbirth, and parenting, '
        'identify 3 trending keywords from this post. '
        'Output format: {"new": keywords}. Do not output anything else.'
    )
    messages = [
        {'role': 'system', 'content': 'You are a data analyst specializing in trend analysis.'},
        {'role': 'user', 'content': prompt}
    ]
    try:
        response = openai.ChatCompletion.create(model=GPT_MODEL, messages=messages)
        return response.choices[0].message['content']
    except Exception as e:
        print(f'API error: {e}')
        return '{}'

In [ ]:
# ── BATCH KEYWORD EXTRACTION ─────────────────────────────────────────────

col_seq, col_word, col_date = [], [], []
col_title, col_contents, col_post_number, col_url, col_views = [], [], [], [], []

for _, row in content_trend.iterrows():
    try:
        str_d = extract_trend(row['TITLE'], row['CONTENTS'])
        d = eval(str_d)
        for keyword in d['new']:
            col_seq.append(row['SEQ'])
            col_word.append(keyword)
            col_date.append(row['WDATE'])
            col_title.append(row['TITLE'])
            col_contents.append(row['CONTENTS'])
            col_post_number.append(row['POST_NUMBER'])
            col_url.append(row['URL'])
            col_views.append(row['VIEWS'])
    except Exception as e:
        print(f'Failed to process row: {e}')
        time.sleep(60)

In [ ]:
# ── BUILD OUTPUT DATAFRAME ───────────────────────────────────────────────

output = pd.DataFrame({
    'SEQ': col_seq,
    'DATE': pd.to_datetime(col_date, format='%Y-%m-%d').strftime('%Y-%m-%d'),
    'KEYWORD': col_word,
    'TITLE': col_title,
    'CONTENTS': col_contents,
    'POST_NUMBER': col_post_number,
    'URL': col_url,
    'VIEWS': col_views
})

# Exclude brand names from keyword results
brands_to_exclude = list(BRAND_KEYWORDS.keys())
output = output[~output['KEYWORD'].isin(brands_to_exclude)]

# Add week start date (Sunday-based)
output['START_WEEK'] = (
    pd.to_datetime(output['DATE'])
    - pd.to_timedelta(pd.to_datetime(output['DATE']).dt.weekday, unit='d')
).dt.strftime('%Y-%m-%d')

output = output[output['CONTENTS'] != '']
output['KEYWORD'] = output['KEYWORD'].str.replace(' ', '')
output = output[['SEQ', 'DATE', 'START_WEEK', 'KEYWORD', 'TITLE', 'CONTENTS',
                 'POST_NUMBER', 'URL', 'VIEWS']]
print(f'Trend keywords extracted: {len(output)}')

In [ ]:
# ── UPLOAD TO DATA WAREHOUSE ─────────────────────────────────────────────

engine = create_engine(URL(
    account=os.environ['SNOWFLAKE_ACCOUNT'],
    user=os.environ['SNOWFLAKE_USER'],
    password=os.environ['SNOWFLAKE_PASSWORD'],
    database=os.environ['SNOWFLAKE_DATABASE'],
    schema=os.environ['SNOWFLAKE_SCHEMA'],
    warehouse=os.environ['SNOWFLAKE_WAREHOUSE'],
))

table_name = 'TREND_TABLE'

with engine.connect() as con:
    output.to_sql(name=table_name, con=con, if_exists='append',
                  method=pd_writer, index=False)

print(f'Uploaded {len(output)} trend keywords to {table_name}')